# 3. PyTorch inference

Load your final checkpoint and recognize already-cropped plates or words. Each notebook has its own configuration/bootstrap cell; no variables from another notebook are required.

Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


In [ ]:
from PIL import Image
from parseq_ocr.inference import TorchRecognizer
from parseq_ocr.data import scan_pairs

recognizer = TorchRecognizer(config)
records = scan_pairs(config["dataset_root"] / "val", config["normalization"])
image_path = Path(records[0]["path"])  # Replace with any cropped image.
with Image.open(image_path) as image:
    image = image.convert("RGB")
print({"ground_truth": records[0]["label"], "prediction": recognizer.infer([image])[0]})
display(image)

The image is resized to the trained height and width with bicubic interpolation and normalized to [-1, 1]. A crop recognizer does not locate plates in full vehicle frames. No Indian-format regex, character substitution, or dictionary correction is applied.